# NYC Taxi — Exploratory Data Analysis
Surge pricing predictor · January 2024 data

In [ ]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

RAW = pathlib.Path('../data/raw.parquet')
df = pd.read_parquet(RAW)
print(df.shape)
df.head()

In [ ]:
# Parse pickup datetime
df['pickup_dt'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['hour'] = df['pickup_dt'].dt.hour
df['dow']  = df['pickup_dt'].dt.dayofweek   # 0=Mon

## Trip demand by hour of day

In [ ]:
hourly = df.groupby('hour').size().reset_index(name='trips')

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(data=hourly, x='hour', y='trips', ax=ax, color='steelblue')
ax.set_title('Trip demand by hour of day (Jan 2024)')
ax.set_xlabel('Hour')
ax.set_ylabel('Number of trips')
plt.tight_layout()
plt.show()

## Trip demand by day of week

In [ ]:
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = df.groupby('dow').size().reset_index(name='trips')
daily['day'] = daily['dow'].map(lambda i: day_labels[i])

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=daily, x='day', y='trips', ax=ax, palette='Blues_d')
ax.set_title('Trip demand by day of week')
ax.set_xlabel('Day')
ax.set_ylabel('Number of trips')
plt.tight_layout()
plt.show()

## Top 10 pickup zones by volume

In [ ]:
top_zones = (
    df['PULocationID']
    .value_counts()
    .head(10)
    .reset_index()
)
top_zones.columns = ['zone_id', 'trips']

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=top_zones, x='zone_id', y='trips', ax=ax, palette='Oranges_r')
ax.set_title('Top 10 pickup zones by trip volume')
ax.set_xlabel('PU Location ID')
ax.set_ylabel('Number of trips')
plt.tight_layout()
plt.show()

## Flagging potential surge hours (top 20% demand periods)

In [ ]:
threshold = hourly['trips'].quantile(0.80)
hourly['surge_flag'] = hourly['trips'] >= threshold

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['tomato' if s else 'steelblue' for s in hourly['surge_flag']]
ax.bar(hourly['hour'], hourly['trips'], color=colors)
ax.axhline(threshold, color='red', linestyle='--', label=f'80th pct ({threshold:,.0f})')
ax.set_title('Surge hours flagged (red = top 20% demand)')
ax.set_xlabel('Hour')
ax.set_ylabel('Number of trips')
ax.legend()
plt.tight_layout()
plt.show()

print('Surge hours:', hourly.loc[hourly['surge_flag'], 'hour'].tolist())